# Publish: Load Order Check & Validation

**Purpose**: Validate row counts, checksums, load order dependencies, and prepare artifacts for consumer-mode bootstrap with comprehensive pre-flight checks.

**Key Features**:
- Dependency graph validation
- Circular dependency detection
- Row count reconciliation (source vs. export vs. target)
- Checksum verification
- Schema compatibility checks
- Load order simulation
- Bootstrap readiness verification
- Rollback preparation

**Validation Checks**:
1. Manifest integrity
2. File existence and accessibility
3. Checksum validation
4. Row count validation
5. Dependency graph validation
6. Load order feasibility
7. Schema compatibility
8. Consumer requirements

In [0]:
import os
import json
import hashlib
from datetime import datetime
from typing import Dict, List, Set, Tuple
from collections import defaultdict, deque
import pandas as pd

# Configuration
CATALOG = "workspace"
WAREHOUSE_SCHEMA = "warehouse"  # Dimensions and Facts
GOLD_SCHEMA = "gold"  # Analytics marts
AUDIT_SCHEMA = "audit"  # Audit tables
VOLUME_PATH = "/Volumes/workspace/publish/snapshots"

# Get latest snapshot or specify one
SNAPSHOT_ID = None  # None = use latest

if SNAPSHOT_ID is None:
    snapshots = [
        f.name for f in dbutils.fs.ls(VOLUME_PATH)
        if f.isDir() and f.name != '/' and not f.name.startswith('.')
    ]
    snapshots.sort(reverse=True)
    SNAPSHOT_ID = snapshots[0] if snapshots else None

if SNAPSHOT_ID is None:
    raise ValueError("No snapshots found!")

SNAPSHOT_DIR = f"{VOLUME_PATH}/{SNAPSHOT_ID}"
print(f"✓ Validating snapshot: {SNAPSHOT_ID}")
print(f"✓ Snapshot directory: {SNAPSHOT_DIR}")

In [0]:
# Load all manifest files
print("\nLoading manifests...")

# Load snapshot manifest
with open(f"{SNAPSHOT_DIR}/snapshot_manifest.json".replace('/Volumes/', '/dbfs/Volumes/'), 'r') as f:
    snapshot_manifest = json.load(f)
print(f"✓ Snapshot manifest loaded: {len(snapshot_manifest.get('load_order', []))} tables")

# Load checksums
with open(f"{SNAPSHOT_DIR}/checksums.json".replace('/Volumes/', '/dbfs/Volumes/'), 'r') as f:
    checksums = json.load(f)
print(f"✓ Checksums loaded: {len(checksums)} files")

# Load metadata
with open(f"{SNAPSHOT_DIR}/metadata.json".replace('/Volumes/', '/dbfs/Volumes/'), 'r') as f:
    metadata = json.load(f)
print(f"✓ Metadata loaded")

# Load schema manifest
try:
    with open(f"{SNAPSHOT_DIR}/schema_manifest.json".replace('/Volumes/', '/dbfs/Volumes/'), 'r') as f:
        schema_manifest = json.load(f)
    print(f"✓ Schema manifest loaded: {len(schema_manifest.get('tables', []))} tables")
except:
    print("⚠️  Schema manifest not found")
    schema_manifest = {"tables": []}

# Load compatibility manifest
try:
    with open(f"{SNAPSHOT_DIR}/compatibility.json".replace('/Volumes/', '/dbfs/Volumes/'), 'r') as f:
        compatibility = json.load(f)
    print(f"✓ Compatibility manifest loaded")
except:
    print("⚠️  Compatibility manifest not found")
    compatibility = {}

In [0]:
def calculate_file_checksum(file_path: str) -> str:
    """Calculate MD5 checksum of file."""
    md5_hash = hashlib.md5()
    local_path = file_path.replace('/Volumes/', '/dbfs/Volumes/')
    
    with open(local_path, 'rb') as f:
        for chunk in iter(lambda: f.read(4096), b""):
            md5_hash.update(chunk)
    
    return md5_hash.hexdigest()

def validate_file_exists(file_path: str) -> bool:
    """Check if file exists."""
    local_path = file_path.replace('/Volumes/', '/dbfs/Volumes/')
    return os.path.exists(local_path)

def get_file_size(file_path: str) -> int:
    """Get file size in bytes."""
    local_path = file_path.replace('/Volumes/', '/dbfs/Volumes/')
    return os.path.getsize(local_path) if os.path.exists(local_path) else 0

def detect_circular_dependencies(dependencies: Dict[str, List[str]]) -> List[List[str]]:
    """
    Detect circular dependencies in the dependency graph.
    Returns list of cycles found.
    """
    def dfs(node, visited, rec_stack, path):
        visited.add(node)
        rec_stack.add(node)
        path.append(node)
        
        for neighbor in dependencies.get(node, []):
            if neighbor not in visited:
                cycle = dfs(neighbor, visited, rec_stack, path.copy())
                if cycle:
                    return cycle
            elif neighbor in rec_stack:
                # Found a cycle
                cycle_start = path.index(neighbor)
                return path[cycle_start:] + [neighbor]
        
        rec_stack.remove(node)
        return None
    
    visited = set()
    cycles = []
    
    for node in dependencies:
        if node not in visited:
            cycle = dfs(node, visited, set(), [])
            if cycle:
                cycles.append(cycle)
    
    return cycles

def topological_sort(dependencies: Dict[str, List[str]]) -> List[str]:
    """
    Perform topological sort on dependency graph.
    Returns sorted list of tables in load order.
    """
    # Build in-degree map
    in_degree = {table: 0 for table in dependencies}
    for table in dependencies:
        for dep in dependencies[table]:
            if dep in in_degree:
                in_degree[dep] += 1
    
    # Initialize queue with tables that have no dependencies
    queue = deque([table for table, degree in in_degree.items() if degree == 0])
    sorted_tables = []
    
    while queue:
        table = queue.popleft()
        sorted_tables.append(table)
        
        # Reduce in-degree for dependent tables
        for other_table, deps in dependencies.items():
            if table in deps:
                in_degree[other_table] -= 1
                if in_degree[other_table] == 0:
                    queue.append(other_table)
    
    # Check if all tables were processed
    if len(sorted_tables) != len(dependencies):
        raise ValueError("Circular dependency detected in topological sort")
    
    return sorted_tables

print("✓ Validation functions loaded")

In [0]:
# Validation Check 1: Manifest Integrity
print("\n" + "="*80)
print("CHECK 1: MANIFEST INTEGRITY")
print("="*80)

integrity_issues = []

# Check required fields
required_fields = ['manifest_version', 'snapshot_id', 'load_order', 'validation']
for field in required_fields:
    if field not in snapshot_manifest:
        integrity_issues.append(f"Missing required field: {field}")
        print(f"❌ Missing field: {field}")
    else:
        print(f"✓ Field present: {field}")

# Check load_order structure
if 'load_order' in snapshot_manifest:
    for idx, item in enumerate(snapshot_manifest['load_order']):
        required_item_fields = ['sequence', 'table', 'file', 'checksum']
        for field in required_item_fields:
            if field not in item:
                integrity_issues.append(f"Load order item {idx} missing field: {field}")
                print(f"❌ Load order item {idx} missing: {field}")

if not integrity_issues:
    print("\n✓ Manifest integrity check PASSED")
else:
    print(f"\n❌ Manifest integrity check FAILED: {len(integrity_issues)} issues")
    for issue in integrity_issues:
        print(f"  - {issue}")

In [0]:
# Validation Check 2: File Existence and Checksums
print("\n" + "="*80)
print("CHECK 2: FILE EXISTENCE AND CHECKSUMS")
print("="*80)

file_issues = []
checksum_issues = []

for item in snapshot_manifest.get('load_order', []):
    table_name = item['table']
    file_path = f"{SNAPSHOT_DIR}/{item['file']}"
    expected_checksum = item['checksum']
    
    print(f"\nValidating: {table_name}")
    
    # Check file exists
    if not validate_file_exists(file_path):
        file_issues.append(f"File not found: {file_path}")
        print(f"  ❌ File not found: {file_path}")
        continue
    else:
        file_size = get_file_size(file_path)
        print(f"  ✓ File exists: {file_size:,} bytes")
    
    # Verify checksum
    print(f"  Calculating checksum...")
    actual_checksum = calculate_file_checksum(file_path)
    
    if actual_checksum != expected_checksum:
        checksum_issues.append(f"{table_name}: checksum mismatch")
        print(f"  ❌ Checksum MISMATCH")
        print(f"    Expected: {expected_checksum}")
        print(f"    Actual:   {actual_checksum}")
    else:
        print(f"  ✓ Checksum verified: {actual_checksum}")

if not file_issues and not checksum_issues:
    print("\n✓ File existence and checksum check PASSED")
else:
    if file_issues:
        print(f"\n❌ File issues: {len(file_issues)}")
        for issue in file_issues:
            print(f"  - {issue}")
    if checksum_issues:
        print(f"\n❌ Checksum issues: {len(checksum_issues)}")
        for issue in checksum_issues:
            print(f"  - {issue}")

In [0]:
# Validation Check 3: Row Count Validation
print("\n" + "="*80)
print("CHECK 3: ROW COUNT VALIDATION")
print("="*80)

row_count_issues = []
total_expected_rows = snapshot_manifest.get('validation', {}).get('total_expected_rows', 0)
total_actual_rows = 0

for item in snapshot_manifest.get('load_order', []):
    table_name = item['table']
    expected_rows = item.get('row_count', 0)
    
    print(f"\n{table_name}:")
    print(f"  Expected rows: {expected_rows:,}")
    
    # Check against Unity Catalog source - determine correct schema
    if table_name.startswith('audit_'):
        table_schema = AUDIT_SCHEMA
    elif table_name.startswith('gold_'):
        table_schema = GOLD_SCHEMA
    elif table_name.startswith(('dim_', 'fact_', 'bridge_')):
        table_schema = WAREHOUSE_SCHEMA
    else:
        print(f"  ⚠️  Cannot determine schema for {table_name}")
        continue
    
    table_full_name = f"{CATALOG}.{table_schema}.{table_name}"
    
    try:
        actual_rows = spark.table(table_full_name).count()
        print(f"  Source rows:   {actual_rows:,}")
        
        if actual_rows != expected_rows:
            row_count_issues.append(f"{table_name}: expected {expected_rows:,}, source has {actual_rows:,}")
            print(f"  ❌ Row count MISMATCH")
        else:
            print(f"  ✓ Row count matches")
        
        total_actual_rows += actual_rows
        
    except Exception as e:
        print(f"  ⚠️  Could not validate: {str(e)}")
        row_count_issues.append(f"{table_name}: validation error")

print(f"\n" + "-"*80)
print(f"Total expected rows: {total_expected_rows:,}")
print(f"Total actual rows:   {total_actual_rows:,}")

if not row_count_issues and total_expected_rows == total_actual_rows:
    print("\n✓ Row count validation PASSED")
else:
    print(f"\n❌ Row count validation FAILED: {len(row_count_issues)} issues")
    for issue in row_count_issues:
        print(f"  - {issue}")

In [0]:
# Validation Check 4: Dependency Graph Validation
print("\n" + "="*80)
print("CHECK 4: DEPENDENCY GRAPH VALIDATION")
print("="*80)

dependency_issues = []

# Build dependency graph from schema manifest
dependency_graph = {}
for table_info in schema_manifest.get('tables', []):
    table_name = table_info['name']
    dependencies = table_info.get('dependencies', [])
    dependency_graph[table_name] = dependencies
    print(f"{table_name}: {len(dependencies)} dependencies")
    if dependencies:
        print(f"  -> {', '.join(dependencies)}")

# Check for circular dependencies
print("\nChecking for circular dependencies...")
cycles = detect_circular_dependencies(dependency_graph)

if cycles:
    print(f"❌ Found {len(cycles)} circular dependency cycle(s):")
    for idx, cycle in enumerate(cycles):
        cycle_str = ' -> '.join(cycle)
        print(f"  Cycle {idx + 1}: {cycle_str}")
        dependency_issues.append(f"Circular dependency: {cycle_str}")
else:
    print("✓ No circular dependencies found")

# Validate topological sort is possible
print("\nValidating load order feasibility...")
try:
    sorted_tables = topological_sort(dependency_graph)
    print(f"✓ Topological sort successful")
    print(f"  Suggested load order: {' -> '.join(sorted_tables[:5])}...")
except ValueError as e:
    print(f"❌ Topological sort failed: {str(e)}")
    dependency_issues.append(f"Cannot determine valid load order: {str(e)}")

# Validate manifest load order matches dependencies
print("\nValidating manifest load order...")
manifest_order = [item['table'] for item in snapshot_manifest.get('load_order', [])]

for idx, table in enumerate(manifest_order):
    deps = dependency_graph.get(table, [])
    
    # Check if all dependencies appear before this table
    for dep in deps:
        if dep in manifest_order:
            dep_idx = manifest_order.index(dep)
            if dep_idx > idx:
                issue = f"{table} depends on {dep}, but {dep} appears later in load order"
                dependency_issues.append(issue)
                print(f"  ❌ {issue}")

if not dependency_issues:
    print("\n✓ Dependency graph validation PASSED")
else:
    print(f"\n❌ Dependency graph validation FAILED: {len(dependency_issues)} issues")
    for issue in dependency_issues:
        print(f"  - {issue}")

In [0]:
# Validation Check 5: Schema Compatibility
print("\n" + "="*80)
print("CHECK 5: SCHEMA COMPATIBILITY")
print("="*80)

schema_issues = []

# Validate schema manifest exists
if not schema_manifest.get('tables'):
    schema_issues.append("Schema manifest is empty or missing")
    print("❌ Schema manifest is empty or missing")
else:
    print(f"✓ Schema manifest contains {len(schema_manifest['tables'])} table definitions")
    
    # Validate each table has required schema fields
    for table_info in schema_manifest['tables']:
        table_name = table_info['name']
        schema_fields = table_info.get('schema_fields', [])
        
        if not schema_fields:
            schema_issues.append(f"{table_name}: schema fields missing")
            print(f"  ❌ {table_name}: No schema fields defined")
        else:
            # Validate field definitions
            for field in schema_fields:
                if 'name' not in field or 'type' not in field:
                    schema_issues.append(f"{table_name}: incomplete field definition")
                    print(f"  ❌ {table_name}: Incomplete field definition")
                    break
            else:
                print(f"  ✓ {table_name}: {len(schema_fields)} fields defined")

if not schema_issues:
    print("\n✓ Schema compatibility check PASSED")
else:
    print(f"\n❌ Schema compatibility check FAILED: {len(schema_issues)} issues")
    for issue in schema_issues:
        print(f"  - {issue}")

In [0]:
# Validation Check 6: Consumer Requirements
print("\n" + "="*80)
print("CHECK 6: CONSUMER REQUIREMENTS")
print("="*80)

requirement_issues = []

# Check compatibility manifest
if not compatibility:
    requirement_issues.append("Compatibility manifest missing")
    print("❌ Compatibility manifest missing")
else:
    print("✓ Compatibility manifest present")
    
    # Check format compatibility
    format_compat = compatibility.get('format_compatibility', {})
    print(f"\nFormat Compatibility:")
    print(f"  CSV version: {format_compat.get('csv_version', 'unknown')}")
    print(f"  Compression: {', '.join(format_compat.get('compression', []))}")
    print(f"  Encoding: {', '.join(format_compat.get('encoding', []))}")
    
    # Check consumer requirements
    consumer_req = compatibility.get('consumer_requirements', {})
    print(f"\nConsumer Requirements:")
    print(f"  Min Python: {consumer_req.get('min_python_version', 'not specified')}")
    
    required_packages = consumer_req.get('required_packages', [])
    print(f"  Required packages: {len(required_packages)}")
    for pkg in required_packages:
        print(f"    - {pkg}")
    
    # Check validation rules
    validation_rules = compatibility.get('validation_rules', {})
    print(f"\nValidation Rules:")
    for rule, value in validation_rules.items():
        print(f"  {rule}: {value}")

if not requirement_issues:
    print("\n✓ Consumer requirements check PASSED")
else:
    print(f"\n❌ Consumer requirements check FAILED: {len(requirement_issues)} issues")
    for issue in requirement_issues:
        print(f"  - {issue}")

In [0]:
# Generate comprehensive validation report
print("\n" + "="*80)
print("VALIDATION SUMMARY REPORT")
print("="*80)

validation_results = {
    "snapshot_id": SNAPSHOT_ID,
    "validated_at": datetime.now().isoformat(),
    "checks": [
        {
            "check_name": "Manifest Integrity",
            "status": "PASS" if not integrity_issues else "FAIL",
            "issues": integrity_issues
        },
        {
            "check_name": "File Existence and Checksums",
            "status": "PASS" if not (file_issues or checksum_issues) else "FAIL",
            "issues": file_issues + checksum_issues
        },
        {
            "check_name": "Row Count Validation",
            "status": "PASS" if not row_count_issues else "FAIL",
            "issues": row_count_issues
        },
        {
            "check_name": "Dependency Graph Validation",
            "status": "PASS" if not dependency_issues else "FAIL",
            "issues": dependency_issues
        },
        {
            "check_name": "Schema Compatibility",
            "status": "PASS" if not schema_issues else "FAIL",
            "issues": schema_issues
        },
        {
            "check_name": "Consumer Requirements",
            "status": "PASS" if not requirement_issues else "FAIL",
            "issues": requirement_issues
        }
    ]
}

# Print summary
print("\nCheck Results:")
total_checks = len(validation_results['checks'])
passed_checks = sum(1 for check in validation_results['checks'] if check['status'] == 'PASS')

for check in validation_results['checks']:
    status_symbol = "✓" if check['status'] == "PASS" else "❌"
    issue_count = len(check['issues'])
    print(f"{status_symbol} {check['check_name']}: {check['status']} ({issue_count} issues)")

print(f"\n" + "-"*80)
print(f"Total Checks: {total_checks}")
print(f"Passed: {passed_checks}")
print(f"Failed: {total_checks - passed_checks}")

overall_status = "PASS" if passed_checks == total_checks else "FAIL"
if overall_status == "PASS":
    print(f"\n✓✓✓ OVERALL STATUS: PASS - Snapshot is ready for consumer bootstrap ✓✓✓")
else:
    print(f"\n❌❌❌ OVERALL STATUS: FAIL - Snapshot has validation issues ❌❌❌")

# Save validation report
validation_report_file = f"{SNAPSHOT_DIR}/validation_report.json"
with open(validation_report_file.replace('/Volumes/', '/dbfs/Volumes/'), 'w') as f:
    json.dump(validation_results, f, indent=2)

print(f"\n✓ Validation report saved: {validation_report_file}")

In [0]:
# Bootstrap Readiness Check
print("\n" + "="*80)
print("BOOTSTRAP READINESS CHECK")
print("="*80)

readiness_checklist = [
    ("Snapshot manifest present", 'snapshot_manifest.json' in [f.name for f in dbutils.fs.ls(SNAPSHOT_DIR)]),
    ("Checksums file present", 'checksums.json' in [f.name for f in dbutils.fs.ls(SNAPSHOT_DIR)]),
    ("Metadata file present", 'metadata.json' in [f.name for f in dbutils.fs.ls(SNAPSHOT_DIR)]),
    ("Schema manifest present", 'schema_manifest.json' in [f.name for f in dbutils.fs.ls(SNAPSHOT_DIR)]),
    ("Compatibility manifest present", 'compatibility.json' in [f.name for f in dbutils.fs.ls(SNAPSHOT_DIR)]),
    ("All data files present", len(file_issues) == 0),
    ("All checksums valid", len(checksum_issues) == 0),
    ("No circular dependencies", len(cycles) == 0),
    ("Load order valid", len(dependency_issues) == 0),
    ("Row counts validated", len(row_count_issues) == 0)
]

readiness_score = sum(1 for _, status in readiness_checklist if status)
total_items = len(readiness_checklist)

print("\nReadiness Checklist:")
for item, status in readiness_checklist:
    status_symbol = "✓" if status else "❌"
    print(f"{status_symbol} {item}")

print(f"\n" + "-"*80)
print(f"Readiness Score: {readiness_score}/{total_items} ({100*readiness_score//total_items}%)")

if readiness_score == total_items:
    print("\n✓✓✓ BOOTSTRAP READY: Snapshot is fully validated and ready for consumer distribution ✓✓✓")
    bootstrap_status = "READY"
else:
    print(f"\n⚠️ BOOTSTRAP NOT READY: {total_items - readiness_score} items need attention")
    bootstrap_status = "NOT_READY"

# Log validation results to audit table
audit_record = spark.createDataFrame([{
    "snapshot_id": SNAPSHOT_ID,
    "validated_at": datetime.now().isoformat(),
    "overall_status": overall_status,
    "bootstrap_status": bootstrap_status,
    "checks_passed": passed_checks,
    "checks_failed": total_checks - passed_checks,
    "readiness_score": f"{readiness_score}/{total_items}",
    "total_issues": sum(len(check['issues']) for check in validation_results['checks'])
}])

audit_record.write.mode("append").saveAsTable(f"{CATALOG}.{AUDIT_SCHEMA}.publish_validation_log")

print(f"\n✓ Validation results logged to {AUDIT_SCHEMA}.publish_validation_log")